# Day 17(M4 Day04) 부록 — LangSmith로 실행 추적하기

**목표**: 본 실습의 로컬 추적(`.stream`, 노드별 실행 시간)으로는 보이지 않는 것 — 프롬프트 원문, 토큰 수, LLM 호출 단위 지연, 실패한 노드의 입력 — 을 LangSmith로 확인한다. 캡스톤(Day 18~20) 필수 요건 5 "LangSmith로 실행을 추적한다"를 여기서 준비한다.

| 절 | 내용 |
| --- | --- |
| 0 | 키 확인과 추적 켜기 — 이 노트북에서만 |
| Part 1 | 연결 확인 — 인증 진단, 첫 추적이 서버에 올라갔는지 |
| Part 2 | 리서치 도우미 그래프 추적 — 로컬 기록과 LangSmith 트리 비교 |
| Part 3 | 오류 대응 — 노드가 실패하면 LangSmith에 무엇이 남는가 |
| Part 4 | 운영 수칙 — 가리기, 끄기 |

**선수**: Day17 본 실습 Part 1·2(Planning, `.stream` 추적, 리서치 도우미). LangSmith 가입과 키 발급은 배포 문서 「외부 API 키 발급」의 LangSmith 절을 따른다.

> ⚠️ 추적을 켜면 **프롬프트와 응답 전체가 LangSmith 서버로 전송**된다. 개인정보·회사 자료·API 키가 들어간 질문은 넣지 않는다. 이 노트북은 연습용 질문만 쓴다.

> **참고:** 다른 노트북은 모두 첫 셀에서 추적을 끈다(`LANGSMITH_TRACING=false`). 이 노트북만 코드에서 켜고, Part 4에서 다시 끈다. `.env`의 `LANGSMITH_TRACING` 값은 바꾸지 않아도 된다. 새로 설치할 패키지는 없다(`langsmith`는 LangChain과 함께 이미 설치돼 있다).

## 0. 키 확인과 추적 켜기

LangSmith 키는 `lsv2_`로 시작한다. 예제 값(`your…`)이 그대로 있으면 추적은 켜진 것처럼 보이지만 서버가 `401`/`403`으로 거절하고, 추적기는 이 실패를 **조용히 삼킨다.** 그래서 켜기 전에 키 모양부터 본다.

프로젝트 이름은 이 노트북에서 `llm-dev-day17`로 고정한다. `.env`에 다른 프로젝트 이름이 있어도 이 노트북의 추적은 여기로 모인다.

In [2]:
import os, re, time, uuid
from dotenv import load_dotenv

load_dotenv()
key = (os.getenv("LANGSMITH_API_KEY") or "").strip()
# 키 값은 출력하지 않는다. 길이와 앞 다섯 글자만 본다
print(f"LANGSMITH_API_KEY: 길이 {len(key)}, 앞 {key[:5]!r}")
if not key.startswith("lsv2_"):
    raise RuntimeError("LangSmith 키가 없거나 예제 값이다. smith.langchain.com에서 키를 발급해 .env에 넣고 커널을 다시 시작한다.")

PROJECT = "llm-dev-day17"
os.environ["LANGSMITH_TRACING"] = "true"          # 이 노트북에서만 켠다
os.environ["LANGSMITH_PROJECT"] = PROJECT
os.environ.pop("LANGCHAIN_TRACING_V2", None)       # 옛 이름의 변수가 false로 남아 있으면 헷갈린다
print("추적 켬 | 프로젝트:", PROJECT)

LANGSMITH_API_KEY: 길이 51, 앞 'lsv2_'
추적 켬 | 프로젝트: llm-dev-day17


## Part 1. 연결 확인

### 1-1. 인증 진단

`Client()`는 환경 변수의 키로 서버에 붙는다. 프로젝트 목록을 한 건 불러와 보면 키가 실제로 통하는지 알 수 있다.

In [2]:
from langsmith import Client

client = Client()
try:
    next(iter(client.list_projects(limit=1)), None)
    print("인증 성공 — 서버 연결과 키 모두 정상")
except Exception as e:
    print("인증 실패:", type(e).__name__, str(e)[:200])

인증 성공 — 서버 연결과 키 모두 정상


### 1-2. 첫 추적 — 서버에 올라갔는지 확인

추적은 **백그라운드로** 전송된다. 셀이 끝나도 아직 안 올라갔을 수 있다. `wait_for_all_tracers()`로 전송을 마친 뒤, 대시보드가 아니라 **API로** 서버에 직접 물어본다. 전송이 끝나도 서버에 **반영되는 데** 몇 초가 더 걸린다. `read_run_when_ready()`는 실행이 끝난 기록이 보일 때까지 1초 간격으로 다시 묻는다. 대시보드에서 안 보일 때 원인이 "전송 실패"인지 "다른 프로젝트를 보고 있음"인지 가를 수 있다.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tracers.langchain import wait_for_all_tracers

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = ChatPromptTemplate.from_template("{q}를 한 문장으로 설명해줘.") | llm

run_id = uuid.uuid4()                      # 이 실행의 이름표. 서버에서 이 번호로 찾는다
# TODO: config에 run_id와 run_name("day17-첫-추적")을 넣어 chain을 실행하고 답을 출력하세요
print(chain.invoke({"q": "RAG"}, config={"run_id": run_id, "run_name": "day17-첫번째-추적"}).content)

# TODO: 남은 추적을 서버로 보내세요 (wait_for_all_tracers)
wait_for_all_tracers()

def find(run, name):
    """트리에서 이름이 name인 run을 모두 찾는다."""
    hits = [run] if run.name == name else []
    for c in run.child_runs or []:
        hits += find(c, name)
    return hits

def read_run_when_ready(run_id, expect=(), tries=30):
    """서버 반영에 몇 초 걸린다. 뿌리 실행이 끝나 있고 expect의 노드가 모두 보일 때까지 1초 간격으로 다시 묻는다."""
    run = None
    for _ in range(tries):
        try:
            run = client.read_run(run_id, load_child_runs=True)
            if run.end_time and all(find(run, name) for name in expect):
                return run
        except Exception:
            pass                         # 아직 서버에 없다
        time.sleep(1)
    if run is None:
        raise RuntimeError("서버에서 run을 찾지 못했다. 키·프로젝트·네트워크를 확인한다.")
    return run                           # 기다려도 덜 올라왔으면 있는 만큼 돌려준다

run = read_run_when_ready(run_id)
print(f"\n서버 기록: {run.name} | 프로젝트 {PROJECT} | 토큰 {run.total_tokens} | {(run.end_time - run.start_time).total_seconds():.2f}초")

RAG(Recall-Augmented Generation)는 정보 검색과 자연어 생성을 결합하여, 외부 데이터베이스에서 관련 정보를 검색한 후 이를 바탕으로 더 정확하고 풍부한 응답을 생성하는 AI 모델입니다.

서버 기록: day17-첫번째-추적 | 프로젝트 llm-dev-day17 | 토큰 67 | 2.25초


In [5]:
client.create_feedback(run_id, key="correctness", score=0.9, comment="출처와 일치")

Feedback(id=UUID('01a11509-c064-7d80-b17c-f2f33acaa5ab'), created_at=datetime.datetime(2026, 10, 7, 6, 25, 23, 43581, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 10, 7, 6, 25, 23, 43581, tzinfo=datetime.timezone.utc), run_id=UUID('6ab38dc9-db94-473c-819c-6770a5468362'), trace_id=None, key='correctness', score=0.9, value=None, comment='출처와 일치', correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

### 관찰

- 서버 기록에 `day17-첫-추적`이 있으면 연결이 끝났다. 브라우저에서 [smith.langchain.com](https://smith.langchain.com) → **Tracing Projects** → `llm-dev-day17`을 열면 같은 실행이 보인다.
- 대시보드에서 안 보이는데 위 셀이 성공했다면, 대개 다른 프로젝트(`default` 등)를 보고 있는 것이다.
- `run_id`를 직접 정해 넘기면 실행과 서버 기록을 1:1로 맞출 수 있다. Part 2·3에서도 이 방법으로 찾는다.
- 이 노트북을 처음 만들 때는 `read_run`을 한 번만 불렀다. 그러자 트리가 **덜 올라온 채로** 읽혀, 그래프 전체 시간이 0초로 나오고 마지막 노드가 빠졌다. 전송(`wait_for_all_tracers`)과 서버 반영은 다르다. 그래서 끝난 기록이 보일 때까지 기다린다.

## Part 2. 리서치 도우미 그래프 추적

본 실습 Part 2의 리서치 도우미(계획 → 검색 → 답변 → 비평 → 정리)를 그대로 가져온다. 다른 점은 두 가지다.

| 바꾼 것 | 이유 |
| --- | --- |
| 지식 저장소를 `InMemoryVectorStore`로 | 이 부록은 DB 없이 돌아가게 한다. 추적 대상은 그래프다 |
| `stream`으로 단계마다 걸린 시간을 잰다 | 로컬 추적. LangSmith 기록과 비교한다 (본 실습 1-3의 `.stream`) |

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

KNOW = [
    "임베딩은 문장을 숫자 벡터로 바꿔 의미를 비교할 수 있게 한다.",
    "RAG는 관련 문서를 찾아 근거로 답해 환각을 줄인다.",
    "LangGraph는 상자(노드)와 화살표(엣지)로 흐름을 제어한다.",
    "체크포인터는 대화 상태를 thread별로 저장해 기억을 잇는다.",
]
store = InMemoryVectorStore.from_texts(KNOW, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = store.as_retriever(search_kwargs={"k": 2})

class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    plan: str
    context: str
    draft: str
    critique: str
    score: int
    n: int

MAX_ITER, PASS = 2, 8
print("지식·State 준비")

In [ ]:
def r_plan(s):
    q = s["messages"][-1].content
    return {"plan": llm.invoke(f"질문: {q}\n답을 위한 3단계 계획을 짧게(번호만).").content}

def r_retrieve(s):
    docs = retriever.invoke(s["messages"][-1].content)
    return {"context": "\n".join(d.page_content for d in docs)}

def r_answer(s):
    guide = f"\n이전 비평: {s['critique']}\n비평을 반영해 고쳐 써라." if s["critique"] else ""
    sys_msg = SystemMessage("너는 친절한 리서치 도우미다. 아래 계획과 지식을 참고해 3문장 이내로 답해라.\n"
                            f"계획:\n{s['plan']}\n지식:\n{s['context']}{guide}")
    return {"draft": llm.invoke([sys_msg] + s["messages"]).content, "n": s["n"] + 1}

def r_critique(s):
    r = llm.invoke(f"질문: {s['messages'][-1].content}\n답: {s['draft']}\n"
                   "0~10점으로 평가하고 개선점을 한 문장. 형식: 점수|개선점").content
    parts = r.split("|", 1)
    digits = "".join(ch for ch in parts[0] if ch.isdigit())     # '8점' → '8'
    return {"score": int(digits), "critique": parts[1].strip() if len(parts) > 1 else ""}

def r_finalize(s):
    return {"messages": [AIMessage(s["draft"])]}

In [ ]:
def route(s):
    if s["score"] >= PASS or s["n"] >= MAX_ITER:
        return "finalize"
    return "revise"

def build_research(critique_fn=r_critique):
    g = StateGraph(ResearchState)
    g.add_node("plan", r_plan)
    g.add_node("retrieve", r_retrieve)
    g.add_node("answer", r_answer)
    g.add_node("critique", critique_fn)
    g.add_node("finalize", r_finalize)
    g.add_edge(START, "plan")
    g.add_edge("plan", "retrieve")
    g.add_edge("retrieve", "answer")
    g.add_edge("answer", "critique")
    g.add_conditional_edges("critique", route, {"revise": "answer", "finalize": "finalize"})
    g.add_edge("finalize", END)
    return g.compile()

research_app = build_research()

def init_state(question):
    return {"messages": [HumanMessage(question)], "plan": "", "context": "", "draft": "",
            "critique": "", "score": 0, "n": 0}
print("리서치 도우미 준비")

### 2-1. 실행 — 이름표 붙이기

`config`에 세 가지를 넣는다. 대시보드에서 실행을 찾고 거를 때 쓴다.

| 키 | 값 | 대시보드에서 |
| --- | --- | --- |
| `run_name` | `research-RAG` | 목록에 보이는 이름 |
| `tags` | `["day17", "research"]` | 태그로 거르기 |
| `metadata` | `{"student": "demo"}` | 수강생·실험 조건 구분 |

In [ ]:
run_id = uuid.uuid4()
cfg = {}   # TODO: run_id, run_name("research-RAG"), tags(["day17", "research"]), metadata({"student": "demo"})를 넣으세요

# 로컬 추적: stream으로 한 단계씩 받으며, 단계 사이에 걸린 시간을 잰다 (본 실습 1-3의 .stream)
local_times = []
answer = ""
prev = time.perf_counter()
for step in research_app.stream(init_state("RAG가 뭐야?"), config=cfg, stream_mode="updates"):
    now = time.perf_counter()
    for node, update in step.items():
        local_times.append((node, round(now - prev, 2)))
        if update and "draft" in update:
            answer = update["draft"]
    prev = now
wait_for_all_tracers()

print("답:", answer)
print()
print("[로컬 추적] 노드별 실행 시간")
for name, sec in local_times:
    print(f"  {name:10} {sec:5.2f}초")

### 2-2. 서버의 실행 트리 읽기

LangSmith는 실행을 **트리**로 저장한다. 그래프 전체가 뿌리이고, 노드가 그 아래, 노드 안의 LLM 호출·검색이 또 그 아래에 달린다. 대시보드에서 보는 것과 같은 트리를 API로 읽어 출력한다.

In [ ]:
root = read_run_when_ready(run_id, expect=("plan", "retrieve", "answer", "critique", "finalize"))

def start_of(run):
    return run.start_time

def show_tree(run, depth=0, max_depth=3):
    sec = (run.end_time - run.start_time).total_seconds() if run.end_time else 0
    tok = f" | 토큰 {run.total_tokens}" if run.total_tokens else ""
    err = f" | 오류 {run.error[:60]}" if run.error else ""
    print(f"{'  ' * depth}- {run.name} [{run.run_type}] {sec:.2f}초{tok}{err}")
    if depth < max_depth:
        children = list(run.child_runs or [])
        children.sort(key=start_of)                  # 먼저 시작한 것부터
        for child in children:
            show_tree(child, depth + 1, max_depth)

show_tree(root)
print(f"\n전체 토큰 {root.total_tokens} (입력 {root.prompt_tokens}, 출력 {root.completion_tokens})")

### 2-3. 노드 안의 프롬프트 원문 보기

로컬 추적은 "critique 노드가 몇 초 걸렸다"까지만 안다. LangSmith는 그 노드가 **LLM에 무엇을 보냈고 무엇을 받았는지**를 남긴다. 비평이 이상할 때 원인을 찾는 곳이 여기다.

In [ ]:
critique_node = None   # TODO: 트리(root)에서 이름이 "critique"인 첫 run을 찾으세요
llm_call = None        # TODO: 그 아래에서 이름이 "ChatOpenAI"인 첫 run을 찾으세요
prompt = llm_call.inputs["messages"][0][0]["kwargs"]["content"]
answer = llm_call.outputs["generations"][0][0]["text"]
print("[critique가 LLM에 보낸 프롬프트]\n" + prompt[:400])
print("\n[LLM의 응답]\n" + answer)

### 관찰 — 로컬 추적과 LangSmith

| 보고 싶은 것 | 로컬 (`.stream`) | LangSmith |
| --- | --- | --- |
| 어느 노드가 몇 번, 어떤 순서로 | 보인다 | 보인다 |
| 노드별 실행 시간 | 보인다 (직접 잰 값) | 보인다 (위 두 표의 시간이 거의 같다) |
| 노드 안의 LLM 호출·검색 단위 시간 | 안 보인다 | 트리로 보인다 |
| 토큰 수(= 비용) | 안 보인다 | 호출마다, 전체 합계 |
| 프롬프트 원문·응답 원문 | 안 보인다 | 보인다 |
| 설정·코드 변경 | 노드마다 기록 코드를 넣어야 한다 | 환경 변수만 켜면 된다 |
| 데이터 전송 | 없다 | 프롬프트가 외부 서버로 간다 |

- 트리에서 토큰을 가장 많이 쓴 노드는 `answer`다(이번 실행 188 / 전체 374). 계획과 검색한 지식을 시스템 메시지에 모두 넣기 때문이다. 비용을 줄이려면 여기부터 본다.
- 이번 실행은 비평 점수가 8점(기준 8)이라 `answer → critique`를 한 번만 돌았다. 점수가 기준에 못 미치면 `answer`·`critique`가 트리에 한 번 더 나오고, 토큰도 그만큼 는다.
- `critique` 아래의 `route`는 조건 엣지의 라우터 함수다. LLM 호출이 없어 시간이 0초다. 라우터를 이름 있는 함수(`def route`)로 만들었기 때문에 트리에도 이름이 보인다. 람다로 만들면 `RunnableCallable`처럼 알아보기 어려운 이름이 나온다.
- 로컬 추적은 가볍고 외부 전송이 없어 **항상 켜 둘 수 있다.** LangSmith는 **원인을 파고들 때** 켠다. 캡스톤에서는 둘 다 쓴다.

## Part 3. 오류 대응 — 노드가 실패하면 무엇이 남는가

본 실습 Part 1의 오류 대응 ②(노드 중간에 예외가 나면)를 LangSmith로 다시 본다. 비평 노드가 모델 응답에서 점수를 찾지 못하는 상황을 만든다. 실제로 모델이 형식(`점수|개선점`)을 지키지 않으면 일어나는 일이다.

In [ ]:
def broken_critique(s):
    # 형식을 지키지 않은 응답을 흉내 낸다: 숫자가 없다
    r = "좋은 답변입니다|예시를 하나 더 들면 좋겠습니다"
    parts = r.split("|", 1)
    digits = "".join(ch for ch in parts[0] if ch.isdigit())     # 숫자가 없으면 빈 문자열
    return {"score": int(digits), "critique": parts[1]}          # int("")에서 오류가 난다

broken_app = build_research(critique_fn=broken_critique)
run_id = uuid.uuid4()
try:
    broken_app.invoke(init_state("임베딩이 뭐야?"), config={"run_id": run_id, "run_name": "research-broken", "tags": ["day17"]})
except Exception as e:
    print("노트북에 보이는 것:", type(e).__name__, "|", e)
wait_for_all_tracers()

In [ ]:
root = read_run_when_ready(run_id, expect=("critique",))
show_tree(root, max_depth=1)
failed = find(root, "critique")[0]
print("\n[실패한 노드가 받은 State]")
print("  draft:", failed.inputs.get("draft", "")[:120])
print("  n:", failed.inputs.get("n"))
print("[오류]", failed.error.strip().splitlines()[-1])

### 관찰

- 노트북에는 `ValueError: invalid literal for int()` 한 줄만 남는다. 어느 노드에서, 어떤 입력으로 났는지는 직접 추적해야 한다.
- LangSmith에는 실패한 `critique` 노드가 빨간 표시(`오류`)로 남고, **그 노드가 받은 State**(답변 초안, 반복 횟수)까지 저장된다. 앞선 `plan`·`retrieve`·`answer`는 정상으로 남아 어디까지 진행됐는지도 보인다.
- 실제 서비스에서는 이 오류가 사용자 질문 수천 건 중 몇 건에서만 난다. 그때 입력을 다시 만들 수 없으므로, 실패한 실행의 입력이 남아 있는 것이 가장 큰 쓸모다.
- 이 오류의 근본 해결은 Day16 본 실습 Part 3-확장처럼 **구조화 출력**으로 점수를 받는 것이다.

## Part 4. 운영 수칙 — 가리기, 끄기

| 상황 | 방법 |
| --- | --- |
| 프롬프트 내용을 서버에 남기면 안 된다 | `.env`에 `LANGSMITH_HIDE_INPUTS=true`, `LANGSMITH_HIDE_OUTPUTS=true`. 트리·시간·토큰은 남고 입력·출력 내용만 비운다 |
| 수강생·실험마다 기록을 나눈다 | 프로젝트 이름(`LANGSMITH_PROJECT`)을 따로 준다. `metadata`로 조건을 적는다 |
| 무료 플랜 한도 | 월 트레이스 수에 한도가 있다. 루프가 도는 그래프를 반복 실행하면 빨리 찬다. 실험이 끝나면 끈다 |
| 추적을 끈다 | 아래 셀 |

In [ ]:
wait_for_all_tracers()                        # 남은 기록을 모두 보낸 뒤
os.environ["LANGSMITH_TRACING"] = "false"      # 이 커널에서 끈다
print("추적 끔. 대시보드:", f"https://smith.langchain.com → Tracing Projects → {PROJECT}")

### 캡스톤 체크리스트

캡스톤 필수 요건 5 "LangSmith로 실행을 추적한다"를 채우려면 발표 때 다음을 보여 준다.

- [ ] 대표 질문 한 건의 실행 트리 (노드 순서, 반복 횟수)
- [ ] 가장 오래 걸린 노드와 그 이유
- [ ] 한 질문에 쓴 토큰 수
- [ ] 실패한 실행 한 건과, 그 입력을 보고 고친 내용

## 확인 문제

1. 로컬 추적(`.stream`)으로는 알 수 없고 LangSmith로만 알 수 있는 것 세 가지는?
2. 추적을 켰는데 대시보드에 아무것도 없을 때, 원인을 가르는 방법은? (1-2)
3. `run_id`·`run_name`·`tags`·`metadata`는 각각 무엇에 쓰는가?
4. 노드가 실패했을 때 LangSmith에 남는 것 중 원인을 찾는 데 가장 중요한 것은?
5. 추적을 켜면 어떤 데이터가 어디로 가는가? 이를 줄이는 방법은?
6. 모든 노트북이 기본으로 추적을 끄는 이유는?